
# 🩺 SkinCheck AI — Explainable Skin Disease Risk Screener

**MCA Data Science Project**
**Author:** Anjali Sharma
**Internship:** Skill Circle, Chandigarh

---

### Objective
Build a Convolutional Neural Network (CNN) that classifies skin lesion
images into 7 diagnostic categories, and pairs every prediction with a
**Grad-CAM** visual explanation — so the model's decision is transparent
rather than a black box.

### Pipeline covered in this notebook
1. Environment setup
2. Synthetic dataset generation (schema-compatible with HAM10000/ISIC)
3. Data loading, exploration & augmentation
4. CNN model architecture
5. Model training
6. Evaluation (accuracy, confusion matrix, classification report)
7. **Grad-CAM explainability**
8. Single-image inference demo
9. Conclusion & next steps

> ⚠️ **Dataset note:** Real dermatology datasets (HAM10000 / ISIC Archive)
> require a Kaggle/ISIC account to download and are not auto-fetchable in a
> hosted notebook environment. This notebook **generates a synthetic
> placeholder dataset** with the exact same class structure
> (`akiec, bcc, bkl, df, mel, nv, vasc`) so the entire pipeline — CNN
> training through Grad-CAM — runs end-to-end and is fully verifiable.
> To use real data, upload HAM10000/ISIC images into the same folder
> structure created in Section 2 and simply re-run Sections 4 onward.


## 1. Environment Setup

In [ ]:
# Install/confirm required libraries (Colab already has most of these)
!pip install -q tensorflow pillow matplotlib scikit-learn pandas numpy


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image, ImageDraw, ImageFilter
from tensorflow.keras import layers, models, callbacks, applications
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Project folders (works both on Colab and locally)
BASE_DIR = "/content/SkinCheck_AI" if os.path.exists("/content") else "./SkinCheck_AI"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
MODELS_DIR = os.path.join(BASE_DIR, "models")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")

for d in [DATASET_DIR, MODELS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project base directory:", BASE_DIR)


## 2. Synthetic Dataset Generation

Generates images across the **standard 7-class HAM10000 taxonomy**:

| Code | Full name | Risk level |
|---|---|---|
| `akiec` | Actinic keratoses / intraepithelial carcinoma | High |
| `bcc` | Basal cell carcinoma | High |
| `bkl` | Benign keratosis-like lesion | Low |
| `df` | Dermatofibroma | Low |
| `mel` | Melanoma | High |
| `nv` | Melanocytic nevus (mole) | Low |
| `vasc` | Vascular lesion | Medium |

Each synthetic image is a procedurally generated "lesion" blob on a skin-tone
background, with **class-specific irregularity, asymmetry, and size**
loosely inspired by the real dermoscopy **ABCDE rule** (Asymmetry, Border,
Color, Diameter, Evolution) — malignant classes get more irregular/
asymmetric borders, benign classes are more regular and symmetric.


In [ ]:
IMG_SIZE = 128

CLASSES = {
    "akiec": {"risk": "High",   "malignant": 1, "color": (194, 150, 130)},
    "bcc":   {"risk": "High",   "malignant": 1, "color": (170, 110, 90)},
    "bkl":   {"risk": "Low",    "malignant": 0, "color": (150, 100, 70)},
    "df":    {"risk": "Low",    "malignant": 0, "color": (140, 90, 75)},
    "mel":   {"risk": "High",   "malignant": 1, "color": (90, 55, 45)},
    "nv":    {"risk": "Low",    "malignant": 0, "color": (160, 120, 95)},
    "vasc":  {"risk": "Medium", "malignant": 0, "color": (180, 90, 90)},
}
CLASS_NAMES = list(CLASSES.keys())

CLASS_FULL_NAMES = {
    "akiec": "Actinic Keratoses / Intraepithelial Carcinoma",
    "bcc": "Basal Cell Carcinoma",
    "bkl": "Benign Keratosis-like Lesion",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic Nevus (Mole)",
    "vasc": "Vascular Lesion",
}

# (irregularity, size_fraction, asymmetry) per class
CLASS_PARAMS = {
    "akiec": (0.55, 0.22, 0.5),
    "bcc":   (0.45, 0.20, 0.4),
    "bkl":   (0.20, 0.28, 0.15),
    "df":    (0.15, 0.15, 0.10),
    "mel":   (0.70, 0.30, 0.65),
    "nv":    (0.18, 0.18, 0.12),
    "vasc":  (0.25, 0.16, 0.20),
}

N_TRAIN, N_VAL, N_TEST = 60, 15, 15


In [ ]:
def make_skin_base(size, skin_tone):
    arr = np.ones((size, size, 3), dtype=np.uint8)
    base = np.array(skin_tone, dtype=np.uint8)
    for c in range(3):
        arr[:, :, c] = base[c]
    noise = np.random.normal(0, 8, (size, size, 3))
    arr = np.clip(arr.astype(int) + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr, "RGB")


def draw_lesion(img, lesion_color, irregularity, size_frac, asymmetry):
    draw = ImageDraw.Draw(img, "RGBA")
    w, h = img.size
    cx, cy = w // 2 + random.randint(-8, 8), h // 2 + random.randint(-8, 8)
    r = int(w * size_frac)

    n_points = random.randint(10, 18)
    points = []
    for i in range(n_points):
        angle = 2 * np.pi * i / n_points
        wobble = 1 + irregularity * random.uniform(-0.4, 0.4)
        asym = 1 + (asymmetry * np.cos(angle) * random.uniform(-0.3, 0.3))
        rad = r * wobble * asym
        x = cx + rad * np.cos(angle)
        y = cy + rad * np.sin(angle)
        points.append((x, y))

    fill = lesion_color + (random.randint(180, 230),)
    draw.polygon(points, fill=fill)

    for _ in range(random.randint(1, 4)):
        bx = cx + random.randint(-r // 2, r // 2)
        by = cy + random.randint(-r // 2, r // 2)
        br = random.randint(3, max(4, r // 3))
        shade = tuple(np.clip(np.array(lesion_color) + random.randint(-40, 40), 0, 255))
        draw.ellipse([bx - br, by - br, bx + br, by + br], fill=shade + (150,))

    return img.filter(ImageFilter.GaussianBlur(radius=1.2))


def generate_image(cls):
    skin_tone = (np.array([224, 172, 150]) + np.random.randint(-15, 15, 3)).tolist()
    img = make_skin_base(IMG_SIZE, skin_tone)
    irregularity, size_frac, asymmetry = CLASS_PARAMS[cls]
    img = draw_lesion(img, CLASSES[cls]["color"], irregularity, size_frac, asymmetry)
    return img


def build_split(split_name, n_per_class, rows):
    for cls in CLASSES:
        out_dir = os.path.join(DATASET_DIR, split_name, cls)
        os.makedirs(out_dir, exist_ok=True)
        for i in range(n_per_class):
            img = generate_image(cls)
            fname = f"{cls}_{split_name}_{i:04d}.jpg"
            img.save(os.path.join(out_dir, fname), quality=90)
            rows.append({
                "image_id": fname, "split": split_name, "dx": cls,
                "risk_level": CLASSES[cls]["risk"], "malignant": CLASSES[cls]["malignant"],
                "age": int(np.clip(np.random.normal(50, 15), 5, 90)),
                "sex": random.choice(["male", "female"]),
                "localization": random.choice(
                    ["back", "face", "chest", "arm", "leg", "scalp", "hand", "foot"]),
            })

print("Generating synthetic dataset...")
rows = []
build_split("train", N_TRAIN, rows)
build_split("val", N_VAL, rows)
build_split("test", N_TEST, rows)

metadata = pd.DataFrame(rows)
metadata.to_csv(os.path.join(DATASET_DIR, "metadata.csv"), index=False)

print(f"Done. {len(metadata)} images generated across {len(CLASSES)} classes.\n")
print(metadata.groupby(["split", "dx"]).size().unstack(fill_value=0))


### 2.1 Preview a few sample images per class

In [ ]:
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(18, 3))
for ax, cls in zip(axes, CLASS_NAMES):
    sample_path = os.path.join(DATASET_DIR, "train", cls, os.listdir(os.path.join(DATASET_DIR, "train", cls))[0])
    img = Image.open(sample_path)
    ax.imshow(img)
    ax.set_title(f"{cls}\n({CLASSES[cls]['risk']} risk)", fontsize=10)
    ax.axis("off")
plt.suptitle("Sample synthetic lesion images per class", y=1.08)
plt.tight_layout()
plt.show()


## 3. Data Loading & Augmentation

In [ ]:
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "train"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    shuffle=True,
    seed=SEED,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "val"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATASET_DIR, "test"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    class_names=CLASS_NAMES,
    shuffle=False,
)


In [ ]:
# Data augmentation - reduces overfitting on a small medical imaging dataset
# by simulating real-world variation in lighting, angle, and zoom.
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

train_ds_aug = train_ds.map(lambda x, y: (augmentation(x, training=True), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds_aug = train_ds_aug.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Class order:", CLASS_NAMES)


## 4. CNN Model Architecture

A CNN built from scratch — 4 convolutional blocks with increasing filter
depth (32 → 64 → 128 → 256), BatchNorm for training stability, and
GlobalAveragePooling (instead of Flatten) which both reduces parameters and
produces cleaner Grad-CAM localization later on.


In [ ]:
def build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=len(CLASS_NAMES)):
    inputs = layers.Input(shape=input_shape, name="input_image")
    x = layers.Rescaling(1.0 / 255)(inputs)

    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu", name="last_conv_layer")(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return models.Model(inputs, outputs, name="SkinCheck_CustomCNN")


model = build_custom_cnn()
model.summary()


### 4.1 (Optional) Transfer-learning variant

Recommended once you plug in the **real** HAM10000/ISIC dataset — pretrained
ImageNet filters transfer well to skin texture/color and converge faster on
limited medical data. Skip this cell if you just want the custom CNN.


In [ ]:
def build_transfer_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=len(CLASS_NAMES), freeze_base=True):
    base_model = applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base_model.trainable = not freeze_base

    inputs = layers.Input(shape=input_shape)
    x = applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="last_conv_layer")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    return models.Model(inputs, outputs, name="SkinCheck_MobileNetV2"), base_model

# Uncomment to use transfer learning instead of the custom CNN:
# model, base_model = build_transfer_model()
# model.summary()


## 5. Model Training

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS = 15
ckpt_path = os.path.join(MODELS_DIR, "skincheck_cnn.keras")

cb = [
    callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_accuracy"),
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3),
]

history = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cb,
    verbose=2,
)

model.save(ckpt_path)
print("Model saved to", ckpt_path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["accuracy"], label="Train Accuracy", marker="o")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy", marker="o")
axes[0].set_title("Model Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history["loss"], label="Train Loss", marker="o")
axes[1].plot(history.history["val_loss"], label="Val Loss", marker="o")
axes[1].set_title("Model Loss"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "training_history.png"), dpi=150)
plt.show()

with open(os.path.join(MODELS_DIR, "class_indices.json"), "w") as f:
    json.dump({i: c for i, c in enumerate(CLASS_NAMES)}, f, indent=2)

print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")


## 6. Evaluation on Test Set

> Note: because this run uses synthetic placeholder data, treat these
> metrics as a **pipeline sanity check**, not a clinical accuracy claim.
> Re-run this section after training on the real HAM10000/ISIC dataset for
> meaningful numbers to report.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()


## 7. Explainability — Grad-CAM

**Grad-CAM** (Selvaraju et al., 2017, *ICCV*) shows which regions of an
image most influenced the model's predicted class. This is the core of the
"Explainable" part of the project — it lets a user verify the model is
actually looking at the lesion, not an irrelevant artifact.

**How it works:**
1. Take gradients of the predicted class score w.r.t. the last conv layer's
   feature maps.
2. Global-average-pool those gradients → an importance weight per channel.
3. Weight each channel of the conv output by its importance, sum, apply
   ReLU, and normalize → the heatmap.
4. Resize and overlay the heatmap on the original image.


In [ ]:
RISK_MAP = {"akiec": "High", "bcc": "High", "mel": "High",
            "vasc": "Medium", "bkl": "Low", "df": "Low", "nv": "Low"}

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="last_conv_layer", pred_index=None):
    grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(last_conv_layer_name).output, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index), predictions.numpy()[0]


def overlay_heatmap(original_img, heatmap, alpha=0.45):
    heatmap_resized = np.uint8(255 * heatmap)
    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]
    jet_heatmap = Image.fromarray(np.uint8(jet_heatmap * 255)).resize(original_img.size)

    original = np.array(original_img).astype(np.float32)
    overlay = np.array(jet_heatmap).astype(np.float32) * alpha + original * (1 - alpha)
    return Image.fromarray(np.uint8(overlay))


def explain_prediction(image_path, model, save_path=None):
    original_img = Image.open(image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    img_array = np.expand_dims(np.array(original_img).astype(np.float32), axis=0)

    heatmap, pred_index, probs = make_gradcam_heatmap(img_array, model)
    pred_class = CLASS_NAMES[pred_index]
    overlay_img = overlay_heatmap(original_img, heatmap)

    result = {
        "predicted_class": pred_class,
        "predicted_class_full": CLASS_FULL_NAMES[pred_class],
        "risk_level": RISK_MAP[pred_class],
        "confidence": float(probs[pred_index]),
        "all_probabilities": {c: float(p) for c, p in zip(CLASS_NAMES, probs)},
    }

    if save_path:
        fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
        axes[0].imshow(original_img); axes[0].set_title("Input Image"); axes[0].axis("off")
        axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM Activation Map"); axes[1].axis("off")
        axes[2].imshow(overlay_img)
        axes[2].set_title(f"Prediction: {result['predicted_class_full']}\nRisk: {result['risk_level']}  |  Confidence: {result['confidence']:.1%}")
        axes[2].axis("off")
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()

    return result, overlay_img


In [ ]:
# Run Grad-CAM on one sample per class from the test set
for cls in CLASS_NAMES:
    class_dir = os.path.join(DATASET_DIR, "test", cls)
    sample_path = os.path.join(class_dir, os.listdir(class_dir)[0])
    save_path = os.path.join(REPORTS_DIR, f"gradcam_{cls}.png")
    result, _ = explain_prediction(sample_path, model, save_path=save_path)
    print(f"[{cls}] predicted={result['predicted_class']} "
          f"risk={result['risk_level']} confidence={result['confidence']:.1%}")


## 8. Single-Image Inference Demo

Run this cell and upload any skin lesion photo (in Colab, this opens a file
picker) to get a full prediction + Grad-CAM explanation, exactly as the
Flask web app would show it.


In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    uploaded = files.upload()
    for fname in uploaded.keys():
        result, overlay_img = explain_prediction(fname, model, save_path=os.path.join(REPORTS_DIR, "demo_gradcam.png"))
        print("\n--- SkinCheck AI Prediction ---")
        print(f"Predicted class : {result['predicted_class']} ({result['predicted_class_full']})")
        print(f"Risk level      : {result['risk_level']}")
        print(f"Confidence      : {result['confidence']:.2%}")
        print("\nAll class probabilities:")
        for c, p in sorted(result['all_probabilities'].items(), key=lambda x: -x[1]):
            print(f"  {c:6s}: {p:.2%}")
else:
    print("Not running in Colab — use 04_gradcam_explainability.py from the full project "
          "zip locally, or run this notebook in Google Colab to enable file upload.")


## 9. Conclusion & Next Steps

**What this notebook demonstrates:**
- A complete, working image classification pipeline (data → CNN → training
  → evaluation → explainability) built and validated end-to-end.
- Grad-CAM explainability integrated directly into the prediction flow,
  making every output visually verifiable rather than a black-box label.

**Limitations (state clearly in your report):**
- Trained on **synthetic placeholder data** here — reported accuracy is a
  pipeline sanity check, not a clinical result.
- Not a diagnostic device; intended purely as a research/screening
  prototype and educational project.

**Recommended next steps:**
1. Replace the synthetic dataset with real HAM10000/ISIC images (same
   folder structure — no code changes needed) and re-run from Section 3.
2. Try the MobileNetV2 transfer-learning variant (Section 4.1) for higher
   accuracy on real data.
3. Add class weighting / focal loss to handle real-world class imbalance
   (benign nevi vastly outnumber melanoma cases in HAM10000).
4. Deploy via the companion Flask app (`app.py` in the full project
   package) for an interactive web-based risk screener.

---
*Project: SkinCheck AI — Explainable Skin Disease Risk Screener*
*MCA Data Science Internship, Skill Circle, Chandigarh*
